In [1]:
# Import & Parameters
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

spark = SparkSession.builder.getOrCreate()

# Path del onfiguration file
config_path = "Files/config_ingestion.csv"


StatementMeta(, 8ec3a6b8-4467-460a-bce1-0b94395b4ff0, 3, Finished, Available, Finished, False)

In [2]:
# Create dataframe for configuration file
df_config = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ";")
    .csv(config_path)
)

display(df_config)

StatementMeta(, 8ec3a6b8-4467-460a-bce1-0b94395b4ff0, 4, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, 190fbb38-1c1f-4e54-9455-d9db6e18d96b)

In [3]:
# Create a filter to select only data settled with CopyMode == "COPY" and Enabled == 1
rows = (
    df_config
    #.filter((df_config.CopyMode == "COPY") & (df_config.Enabled == 1)) # se volessi pushare solo quelle in COPY
    .filter(df_config.SourceType != "SQL Server") # Escluso SQL Server poichè sono già pushate nelle Tables nella pipeline
    .filter(df_config.Enabled == 1) # Pushando anche la shortcut
    .collect()
)

print(f"Files to process: {len(rows)}")

StatementMeta(, 8ec3a6b8-4467-460a-bce1-0b94395b4ff0, 5, Finished, Available, Finished, True)

Files to process: 8


In [4]:
# Sanitize  data to guarantee Delta tables constraint
import re

def sanitize_column_names(df):
    """
    Sostituisce spazi e caratteri non validi per Delta con underscore.
    Caratteri vietati da Delta: spazio , ; { } ( ) \n \t =
    """
    new_columns = []
    for col_name in df.columns:
        clean_name = re.sub(r'[ ,;{}()\n\t=]', '_', col_name)
        # Rimuove eventuali underscore multipli consecutivi generati dalla sostituzione
        clean_name = re.sub(r'_+', '_', clean_name)
        # Rimuove underscore iniziali/finali
        clean_name = clean_name.strip('_')
        new_columns.append(clean_name)
    
    for old_name, new_name in zip(df.columns, new_columns):
        if old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)
    
    return df

StatementMeta(, c9279331-6b9b-4c48-8eb5-2626307fcab8, 6, Finished, Available, Finished, False)

In [5]:
# Trasnformation in Delta Tables
results = []

for row in rows:
    source_folder   = row["DestinationFolder"]    # Folders in Files (crm/erp)
    object_name     = row["ObjectName"]           # File name, es. customers.csv
    dest_table      = row["DestinationTable"]     # target table name
    file_format     = row["FileFormat"].lower()
    load_type       = row["LoadType"]

    source_path = f"Files/{source_folder}/{object_name}"
    target_table = dest_table  # Written in Tables/, root del lakehouse

    print(f"Processing: {source_path} -> Tables/{target_table}")

    try:
        # Reading data from Files
        if file_format == "csv":
            df = (
                spark.read
                .option("header", "true")
                .option("inferSchema", "true")
                .csv(source_path)
            )
        elif file_format == "parquet":
            df = spark.read.parquet(source_path)
        elif file_format == "json":
            # 1. Proviamo a leggere il JSON come standard (JSON Lines / NDJSON)
            try:
                df = spark.read.json(source_path)
                # Forziamo Spark a verificare lo schema con un cache e un'azione rapida
                df.cache()
                if len(df.columns) == 1 and df.columns[0] == "_corrupt_record":
                    raise Exception("Formato NDJSON non valido, provo con multiline...")
            except Exception:
                # 2. Se fallisce, proviamo con l'opzione multiLine (classico JSON strutturato)
                df = (
                    spark.read
                    .option("multiLine", "true")
                    .json(source_path)
                )
                df.cache()
        else:
            raise ValueError(f"FileFormat not manageable: {file_format}")
        
        # Sanitizza nomi colonna per compatibilità Delta
        df = sanitize_column_names(df)

        # Additional columns for audit
        df = (
            df
            .withColumn("_ingestion_timestamp", current_timestamp())
            .withColumn("_source_file", lit(object_name))
        )

        # Write as Delta Table
        write_mode = "overwrite" if load_type == "FULL" else "append"

        (
            df.write
            .format("delta")
            .mode(write_mode)
            .option("overwriteSchema", "true" if write_mode == "overwrite" else "false")
            .saveAsTable(target_table)
        )

        results.append({"table": target_table, "status": "SUCCESS", "rows": df.count()})
        print(f"  ✔ OK — {df.count()} righe scritte in {target_table}")

    except Exception as e:
        results.append({"table": target_table, "status": "FAILED", "error": str(e)})
        print(f"  ✘ ERROR on {target_table}: {e}")

StatementMeta(, c9279331-6b9b-4c48-8eb5-2626307fcab8, 7, Finished, Available, Finished, False)

Processing: Files/crm/customers.csv -> Tables/bronze_customers
  ✔ OK — 30000 righe scritte in bronze_customers
Processing: Files/crm/legacy_customers.csv -> Tables/bronze_legacy_customers
  ✔ OK — 12500 righe scritte in bronze_legacy_customers
Processing: Files/erp/suppliers.csv -> Tables/bronze_suppliers
  ✔ OK — 500 righe scritte in bronze_suppliers
Processing: Files/erp/inventory.csv -> Tables/bronze_inventory
  ✔ OK — 10000 righe scritte in bronze_inventory
Processing: Files/shortcut_adls/products.csv -> Tables/bronze_products
  ✔ OK — 10000 righe scritte in bronze_products
Processing: Files/product_catalog/product_catalog.json -> Tables/bronze_product_catalog
  ✔ OK — 10000 righe scritte in bronze_product_catalog
Processing: Files/marketing/marketing_campaigns.csv -> Tables/bronze_marketing_campaigns
  ✔ OK — 1000 righe scritte in bronze_marketing_campaigns
Processing: Files/ecommerce/web_logs.json -> Tables/bronze_web_logs
  ✔ OK — 1150314 righe scritte in bronze_web_logs
Proces

In [6]:
# Check Failure
df_results = spark.createDataFrame(results)
display(df_results)

# Se vuoi far fallire la pipeline in caso di errori:
failed = [r for r in results if r["status"] == "FAILED"]
if failed:
    raise Exception(f"{len(failed)} tabelle fallite: {[f['table'] for f in failed]}")

StatementMeta(, c9279331-6b9b-4c48-8eb5-2626307fcab8, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 77f6ee63-77df-4443-af71-82bca563f60c)

Exception: 1 tabelle fallite: ['bronze_reviews']